# Voronoi Mosaic Generator

Generate Voronoi mosaics from images with different seed distributions (random, grid-jittered, edge-weighted) and rendering styles (flat average, dominant color). Outputs the mosaic and a cell-boundary overlay.

In [ ]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt

# Make the repo's tools importable from notebooks/.
sys.path.insert(0, os.path.abspath(os.path.join('..', 'tools')))
import halftone_common as hc

def demo_image(size=256):
    """A synthetic test image: smooth ramp + gradient disk, for when you
    don't have a photo handy. Replace with hc.load_image('path.jpg')."""
    y, x = np.mgrid[0:size, 0:size] / size
    ramp = x
    disk = np.clip(1.2 - 2.0 * np.hypot(x - 0.5, y - 0.5), 0, 1)
    g = np.clip(0.5 * ramp + 0.5 * disk, 0, 1)
    return np.stack([g, np.roll(g, 20, 0), 1 - g], axis=-1)

img = demo_image()
plt.figure(figsize=(4,4)); plt.imshow(img); plt.title('source'); plt.axis('off'); plt.show()


In [ ]:
import mosaic_tile_generator as mtg

## Random vs edge-weighted seeds
Uniform random seeds give an even abstract mosaic. Placing more seeds where the image has detail makes cell boundaries hug edges.

In [ ]:
h, w = img.shape[:2]
rng = np.random.default_rng(0)

# Edge map (gradient magnitude of luma).
g = hc.to_gray(img)
gx = np.abs(np.diff(g, axis=1, prepend=g[:, :1]))
gy = np.abs(np.diff(g, axis=0, prepend=g[:1, :]))
edge = gx + gy
edge /= edge.sum()

def weighted_seeds(n):
    idx = rng.choice(h*w, size=n, p=edge.ravel())
    return idx % w, idx // w

def voronoi_from_points(sx, sy):
    ys, xs = np.mgrid[0:h, 0:w]
    best = np.full((h, w), np.inf); lab = np.zeros((h, w), int)
    for i in range(len(sx)):
        d = (xs - sx[i])**2 + (ys - sy[i])**2
        c = d < best; best = np.where(c, d, best); lab = np.where(c, i, lab)
    return lab

n = 800
rx, ry = rng.integers(0, w, n), rng.integers(0, h, n)
lab_rand = voronoi_from_points(rx, ry)
ex, ey = weighted_seeds(n)
lab_edge = voronoi_from_points(ex, ey)

m_rand = mtg._region_mean(img, lab_rand, lab_rand.max()+1, dominant=False)
m_edge = mtg._region_mean(img, lab_edge, lab_edge.max()+1, dominant=False)
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(m_rand); ax[0].set_title('random seeds'); ax[0].axis('off')
ax[1].imshow(m_edge); ax[1].set_title('edge-weighted seeds'); ax[1].axis('off')
plt.show()

## Stained-glass styling
Add dark lead lines along cell boundaries.

In [ ]:
glass = mtg.apply_grout(m_edge, lab_edge, width=2)
plt.figure(figsize=(6,6)); plt.imshow(glass); plt.title('stained glass'); plt.axis('off'); plt.show()

## Boundary overlay
Export just the cell edges as a separate layer.

In [ ]:
e = np.zeros(lab_edge.shape, bool)
e[:-1,:] |= lab_edge[:-1,:] != lab_edge[1:,:]
e[:,:-1] |= lab_edge[:,:-1] != lab_edge[:,1:]
plt.figure(figsize=(6,6)); plt.imshow(e, cmap='gray'); plt.title('cell boundaries'); plt.axis('off'); plt.show()

Seed count controls abstraction: few seeds = bold cubist blocks, many seeds = faithful but faceted. Try `n` from 200 to 5000.